In [10]:
#import packages 

import numpy as np
from scipy.constants import h, c, k
from scipy.integrate import quad
from scipy.interpolate import interp1d
import pandas as pd 
import matplotlib.pyplot as plt
from pathlib import Path
import os


In [11]:
# load the dataset 

# === SETTINGS ===
date_str = "20250706" # Date in YYYYMMDD format
ID = "A4"             # Station ID
action_number = "17"  # 17 for Halobates

# load the dataset 

dir_path = Path.cwd()  # current working directory


rad_dir = dir_path.parent / "postproc/uncorrected_radiances"

rad_file= rad_dir / f"{date_str}_{ID}_{action_number}_IR_radiance.csv"
df_rad = pd.read_csv(rad_file, sep=",", decimal=",")

In [12]:
# === Define constants ===
lambda_min = 8     # micrometers
lambda_max = 14    # micrometers
# Define your emissivity range
emissivity_values = np.arange(0.96, 0.991, 0.01)  


# === Define temperature range for lookup tables ===
T_min = 280 # Kelvin
T_max = 310 # Kelvin
n_values = (T_max - T_min) * 1000 # 0.001 K steps

In [18]:
df_rad.head()
L_measured_array=df_rad["Rad_Water"]
L_sky_array=df_rad["Rad_Sky"]

In [14]:
# === Planck function ===
def planck_lambda(T, lam):
    """Spectral radiance [W/m²/sr/μm] at temperature T [K] and wavelength λ [μm]"""
    lam_m = lam * 1e-6  # convert μm to meters
    return (2*h*c**2 / lam_m**5) / (np.exp(h*c / (lam_m * k * T)) - 1)

# === Convert spectral radiance to band-integrated radiance ===
def band_integrated_radiance(T, lam1=lambda_min, lam2=lambda_max):
    """Band-integrated Planck radiance over [λ1, λ2] for temperature T"""
    result, _ = quad(lambda l: planck_lambda(T, l), lam1, lam2)
    return result

In [15]:
# === Step 1:Generate lookup table of temperature-band intergated radiance values ===
T_range = np.linspace(T_min, T_max + 1, n_values)  # Kelvin
radiance_lookup = np.array([band_integrated_radiance(T) for T in T_range])

# Interpolation function to invert Planck's law
radiance_to_temp = interp1d(radiance_lookup, T_range, bounds_error=False, fill_value="extrapolate")

In [17]:



# Prepare an empty DataFrame to hold Time column first
df_sst_all = pd.DataFrame()
df_sst_all["Time"] = df_rad["Time"].values

# Loop over each emissivity value
for emissivity in emissivity_values:

    # Step 3: Correct for reflection and emissivity
    L_emit_array = (L_measured_array - (1 - emissivity) * L_sky_array) / emissivity

    # Step 4: Invert radiance to get SST
    T_sst_array = radiance_to_temp(L_emit_array)

    # Convert back to Celsius
    T_sst_array_C = T_sst_array - 273.15

    # Add SST column for this emissivity to the df_sst_all dataframe
    col_name = f"SST_{emissivity:.2f}"
    df_sst_all[col_name] = T_sst_array_C

    print(f"SST calculated for emissivity {emissivity:.2f}")

# === Save to CSV for analysis ===
output_file = "sst_emissivity_range.csv"
df_sst_all.to_csv(output_file, index=False)
print(f"All SST values for emissivity range saved to {output_file}.")


TypeError: can't multiply sequence by non-int of type 'float'